# Kuku Promo Finder — Colab runner

Run this notebook on a fresh show input. Upload the repository ZIP and screenplay ZIP. Model names and free-tier limits change; verify the model ID and quota in your provider console before execution. API key is read with `getpass` and is not written into the notebook. The notebook runs two independent passes for stability measurement.

In [ ]:
!pip -q install google-genai openai anthropic

In [ ]:
from google.colab import files
import os, zipfile, shutil, getpass, json, time, hashlib, csv
from pathlib import Path
print("Upload the project repository ZIP (without the private input pack).")
repo_upload = files.upload()
repo_zip = next(iter(repo_upload))
WORK = Path('/content/kuku-work')
if WORK.exists(): shutil.rmtree(WORK)
WORK.mkdir()
with zipfile.ZipFile(repo_zip) as z: z.extractall(WORK)
repo_dirs=[p for p in WORK.iterdir() if p.is_dir()]
REPO=repo_dirs[0] if repo_dirs else WORK
print("Repository:", REPO)
print("Now upload the Kuku screenplay ZIP. Keep it private; do not publish it.")
script_upload=files.upload()
SCRIPT_ZIP=Path('/content/screenplays.zip')
SCRIPT_ZIP.write_bytes(next(iter(script_upload.values())))
INPUT=Path('/content/screenplays'); INPUT.mkdir(exist_ok=True)
with zipfile.ZipFile(SCRIPT_ZIP) as z: z.extractall(INPUT)
print('Screenplay text files:',len(list(INPUT.rglob('*.txt'))))

In [ ]:
import sys
sys.path.insert(0,str(REPO/'src'))
from parser import parse_path
from pathlib import Path
paths=sorted(INPUT.rglob('*.txt'))
episodes=[parse_path(p) for p in paths]
print('Episodes:',len(episodes),'Timed beats:',sum(len(e['beats']) for e in episodes))
print('Parser warnings:',sum(len(e['warnings']) for e in episodes))
for e in episodes:
    if e['warnings']: print(e['episode'],e['warnings'])

## Choose provider and credentials
Use a model ID currently available to your account. Gemini is the suggested default; Groq or Anthropic can be selected if configured. The model is replaceable without changing parsing or validation logic.

In [ ]:
provider=input('Provider [gemini/groq/anthropic/openai] (default gemini): ').strip().lower() or 'gemini'
model=input('Exact model ID from provider console (required): ').strip()
if not model: raise ValueError('Enter a current model ID; do not rely on a guessed default.')
env_key={'gemini':'GEMINI_API_KEY','groq':'GROQ_API_KEY','anthropic':'ANTHROPIC_API_KEY','openai':'OPENAI_API_KEY'}[provider]
os.environ[env_key]=getpass.getpass(f'Paste {env_key}: ')
if provider=='groq': os.environ['OPENAI_BASE_URL']='https://api.groq.com/openai/v1'
print('Provider configured:',provider,'Model:',model)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import pathlib
DRIVE_OUT = pathlib.Path('/content/drive/MyDrive/kuku_raw_outputs')
DRIVE_OUT.mkdir(parents=True, exist_ok=True)
print('Saving progress to Google Drive at:', DRIVE_OUT)
print('If Colab disconnects, just reconnect, re-run cells 1-5, then re-run this cell and cell 6/7 again -- already-done episodes will be skipped.')

In [ ]:
from llm_pipeline import run_llm
OUT=DRIVE_OUT
summary1, ranked1=run_llm(episodes,provider,model,str(OUT),'run1')
print(json.dumps(summary1,indent=2))
print('Top-15 promos:',len(ranked1))

In [ ]:
# Independent second pass for stability; do not reuse first-run model responses.
summary2, ranked2=run_llm(episodes,provider,model,str(OUT),'run2')
print(json.dumps(summary2,indent=2))

In [ ]:
def signature(c): return (c['episode'],c['start_beat'],c['end_beat'])
a=[signature(x) for x in ranked1[:10]]; b=[signature(x) for x in ranked2[:10]]
intersection=len(set(a)&set(b)); union=len(set(a)|set(b))
stability={'top10_overlap_count':intersection,'top10_jaccard':round(intersection/union,3) if union else 0,'run1_top10':a,'run2_top10':b}
(OUT/'stability.json').write_text(json.dumps(stability,indent=2))
print(json.dumps(stability,indent=2))

## Export raw outputs
These are unedited pipeline outputs. Keep the source screenplay ZIP private and do not include it in the submission.

In [ ]:
zip_path=Path('/content/raw_outputs.zip')
with zipfile.ZipFile(zip_path,'w',zipfile.ZIP_DEFLATED) as z:
    for f in OUT.rglob('*'):
        if f.is_file(): z.write(f,f.relative_to(OUT))
files.download(str(zip_path))